# Análisis exploratorio de las medidas

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ana-perez/python-ana-perez/blob/main/notebooks/eda.ipynb)

**Preguntas**: ¿cuántas medidas faltan, y en qué etiquetas? ¿Cómo se reparten los valores de cada etiqueta? ¿Hay medidas fuera de lo normal? ¿Difieren las etiquetas entre sí?

**Datos**: `instantanea.csv`, escrita por `observatorio extraer instantanea.csv --carga 1` desde PostgreSQL. El notebook no lee la base de datos: lee esta instantánea y comprueba su huella, así da lo mismo hoy que dentro de un año.

**Para repetirlo**: *Kernel → Restart Kernel and Run All Cells*. En Colab, sube antes `instantanea.csv` con el panel de archivos.

In [ ]:
import sys

# Solo en Colab: instala las versiones de uv.lock. En tu máquina y en Docker ya las tienes.
if "google.colab" in sys.modules:
    import subprocess

    subprocess.run(
        [
            "uv",
            "pip",
            "install",
            "--system",
            "pandas==3.1.0",
            "matplotlib==3.11.2",
            "scipy==1.18.1",
        ],
        check=True,
    )

In [ ]:
import hashlib
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats

SNAPSHOT = Path("instantanea.csv")
# La huella que escribió `observatorio extraer`. Si el archivo cambia, el notebook se para.
EXPECTED = "3193779438eecdede9910a3cbc922f05a883e69a8a92fd4ef9a33e482d636c65"
assert hashlib.sha256(SNAPSHOT.read_bytes()).hexdigest() == EXPECTED, (
    "no es la instantánea analizada"
)

# keep_default_na=False: una etiqueta "NA" o "null" sigue siendo texto; solo un valor vacío falta.
frame = pd.read_csv(SNAPSHOT, keep_default_na=False, na_values={"valor": [""]})
frame.shape

## 1. Forma y tipos

In [ ]:
frame.dtypes

In [ ]:
frame.head()

## 2. Lo que falta

In [ ]:
frame["valor"].isna().groupby(frame["etiqueta"]).agg(["sum", "mean"])

Falta un 5 % de los valores, repartido por igual entre las etiquetas.

## 3. Cómo se reparten

In [ ]:
frame.groupby("etiqueta")["valor"].describe()

In [ ]:
figure, axes = plt.subplots(figsize=(8, 4))
for label, group in frame.groupby("etiqueta"):
    axes.hist(group["valor"].dropna(), bins=20, histtype="step", label=label)
axes.set_xlabel("valor")
axes.set_ylabel("medidas")
axes.legend()

Los valores de cada etiqueta se reparten por igual entre 0 y 80.

## 4. Fuera de lo normal

In [ ]:
q1, q3 = frame["valor"].quantile([0.25, 0.75])
spread = q3 - q1
outside = frame[(frame["valor"] < q1 - 1.5 * spread) | (frame["valor"] > q3 + 1.5 * spread)]
len(outside)

La regla de Tukey no marca ninguna medida.

## 5. Niveles por etiqueta

In [ ]:
pd.crosstab(frame["etiqueta"], frame["nivel"], margins=True)

## 6. ¿Difieren las etiquetas?

In [ ]:
groups = [group["valor"].dropna() for _, group in frame.groupby("etiqueta")]
stats.f_oneway(*groups)

## Conclusiones

- Faltan un 5 % de los valores, sin preferencia por ninguna etiqueta.
- Ninguna medida queda fuera de lo normal.
- Las etiquetas no difieren entre sí: los datos son generados al azar.

No puede responder por qué faltan los valores, ni si las medidas siguen igual fuera de esta carga.